# AlexNet

## 0. 基本认识 AlexNet Overview

AlexNet 是 2012 年 ILSVRC 图像识别竞赛中的代表性深度卷积神经网络。它让 CNN 在大规模图像分类任务上取得明显优势，也推动了深度学习在计算机视觉中的广泛使用。

课件中给出的背景可以概括为：

- LeNet 在小规模数据集上效果较好，但在更大、更真实的数据集上还不够强。
- 2012 年 AlexNet 在 ImageNet 大规模视觉识别挑战赛中取得冠军。
- AlexNet 比 LeNet 更深，参数更多，使用了 ReLU、Dropout、数据增强、最大池化和 LRN 等技术。

简单理解：AlexNet 是把 LeNet 的卷积分类思想扩展到更大图像、更深网络和更大数据集上的一个关键模型。

---

## 1. 网络结构 Network Architecture

### 1.1 和 LeNet 的关系

AlexNet 和 LeNet 的整体思路相似，都是先用卷积层提取图像特征，再用全连接层完成分类。但 AlexNet 有几个明显变化：

| 对比项 | LeNet | AlexNet |
| --- | --- | --- |
| 网络规模 | 较小 | 更深、更宽，参数更多 |
| 卷积层数量 | 2 个 | 5 个 |
| 全连接层 | 3 个 | 2 个隐藏全连接层 + 1 个输出层 |
| 激活函数 | sigmoid / tanh | ReLU |
| 池化方式 | 平均池化 | 最大池化 |
| 适用场景 | 手写数字识别 | 大规模图像分类 |

### 1.2 课件中的 AlexNet 结构

课件参数详解采用的是 ImageNet 口径：输入为 $3 \times 227 \times 227$，最后输出 1000 类。

| 顺序 | 层 | 主要参数 | 输入尺寸 | 输出尺寸 |
| --- | --- | --- | --- | --- |
| 1 | Input | RGB 图像 | - | $3 \times 227 \times 227$ |
| 2 | Conv1 | $11 \times 11$，96 个卷积核，stride=4 | $3 \times 227 \times 227$ | $96 \times 55 \times 55$ |
| 3 | MaxPool1 | $3 \times 3$，stride=2 | $96 \times 55 \times 55$ | $96 \times 27 \times 27$ |
| 4 | Conv2 | $5 \times 5$，256 个卷积核，padding=2 | $96 \times 27 \times 27$ | $256 \times 27 \times 27$ |
| 5 | MaxPool2 | $3 \times 3$，stride=2 | $256 \times 27 \times 27$ | $256 \times 13 \times 13$ |
| 6 | Conv3 | $3 \times 3$，384 个卷积核，padding=1 | $256 \times 13 \times 13$ | $384 \times 13 \times 13$ |
| 7 | Conv4 | $3 \times 3$，384 个卷积核，padding=1 | $384 \times 13 \times 13$ | $384 \times 13 \times 13$ |
| 8 | Conv5 | $3 \times 3$，256 个卷积核，padding=1 | $384 \times 13 \times 13$ | $256 \times 13 \times 13$ |
| 9 | MaxPool3 | $3 \times 3$，stride=2 | $256 \times 13 \times 13$ | $256 \times 6 \times 6$ |
| 10 | Flatten | 展平 | $256 \times 6 \times 6$ | $9216$ |
| 11 | FC1 | Linear | $9216$ | $4096$ |
| 12 | FC2 | Linear | $4096$ | $4096$ |
| 13 | FC3 | Linear | $4096$ | $1000$ |

### 1.3 主线结构

AlexNet 的主线可以写成：

```text
Conv -> ReLU -> MaxPool
Conv -> ReLU -> MaxPool
Conv -> ReLU -> Conv -> ReLU -> Conv -> ReLU -> MaxPool
Flatten -> FC -> ReLU -> Dropout -> FC -> ReLU -> Dropout -> FC
```

卷积层负责逐步提取图像特征，最大池化负责压缩空间尺寸，全连接层负责把高维特征映射到最终类别。

---

## 2. 网络参数计算 Parameter Calculation

### 2.1 卷积和池化输出尺寸

卷积或池化后的空间尺寸可以用同一个公式理解：

$$
O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1
$$

其中：

- $I$：输入尺寸。
- $P$：padding 大小。
- $K$：卷积核或池化核大小。
- $S$：stride 步幅。
- $O$：输出尺寸。

### 2.2 关键尺寸计算

第一层卷积：输入 $227 \times 227$，卷积核 $11 \times 11$，stride=4，padding=0。

$$
O = \left\lfloor \frac{227 - 11}{4} \right\rfloor + 1 = 55
$$

所以 Conv1 输出为 $96 \times 55 \times 55$。

第一次最大池化：输入 $55 \times 55$，池化核 $3 \times 3$，stride=2。

$$
O = \left\lfloor \frac{55 - 3}{2} \right\rfloor + 1 = 27
$$

所以 MaxPool1 输出为 $96 \times 27 \times 27$。

后面几层通过 padding 保持尺寸，或通过池化降低尺寸：

```text
27 x 27 -- Conv5x5, padding=2 --> 27 x 27
27 x 27 -- MaxPool3x3, stride=2 --> 13 x 13
13 x 13 -- Conv3x3, padding=1 --> 13 x 13
13 x 13 -- MaxPool3x3, stride=2 --> 6 x 6
```

最后一次池化后得到 $256 \times 6 \times 6$，展平后长度为：

$$
256 \times 6 \times 6 = 9216
$$

这就是源码中 `nn.Linear(6*6*256, 4096)` 的原因。

---

## 3. 关键改进 Key Techniques

### 3.1 ReLU 激活函数

AlexNet 使用 ReLU 替代传统的 sigmoid 或 tanh：

$$
ReLU(x) = \max(0, x)
$$

主要作用：

- 计算更简单。
- 收敛速度更快。
- 一定程度上缓解 sigmoid / tanh 容易梯度变小的问题。

### 3.2 最大池化 Max Pooling

LeNet 中常用平均池化，AlexNet 使用最大池化。最大池化会保留局部区域中响应最强的特征：

- 有助于保留更明显的边缘、纹理和局部模式。
- 降低特征图尺寸，减少后续计算量。
- 课件中强调 AlexNet 使用了重叠最大池化，即池化步幅小于池化核尺寸，例如 $3 \times 3$ 池化、stride=2。

### 3.3 Dropout

Dropout 在训练时随机让一部分隐藏神经元临时失活，减少模型对某些神经元的依赖。

训练时大致流程：

1. 随机临时删除一部分隐藏神经元。
2. 用剩余网络完成前向传播和反向传播。
3. 更新未被删除神经元对应的参数。
4. 下一批数据重新随机选择失活神经元。

作用：提高泛化能力，减少过拟合。AlexNet 在前两个全连接层后使用 Dropout。

### 3.4 数据增强 Data Augmentation

课件中提到的数据增强包括：

- 水平翻转：把图像左右翻转，增加样本变化。
- 随机裁剪：从原图中随机裁出不同区域，提升模型对位置变化的适应能力。
- PCA 颜色扰动：对 RGB 三通道做扰动，增强模型对颜色变化的鲁棒性。

数据增强的目的不是改变标签，而是在标签不变的前提下制造更多训练样本，减少过拟合。

### 3.5 LRN 局部响应归一化

LRN Local Response Normalization 的作用是对局部响应做归一化，让较大的响应相对更突出，增强局部对比度。

课件中给出的理解是：

- 手动设置超参数 $k, \alpha, \beta, n$。
- 对某个通道位置的值，结合附近通道的响应进行归一化。
- AlexNet 中使用 LRN 带来了一定准确率提升。

现在很多 CNN 已经较少使用 LRN，常见替代方式是 BatchNorm 等归一化方法。

---

## 4. PyTorch 实战版 PyTorch Implementation

### 4.1 课件原理版和源码实战版的差别

课件参数详解按 ImageNet 版本说明：

```text
输入: 3 x 227 x 227
输出: 1000 类
```

配套源码为了在 FashionMNIST 上训练，做了简化：

```text
输入: 1 x 227 x 227
输出: 10 类
```

也就是说，源码保留了 AlexNet 的主干结构，但把输入通道和输出类别改成适合 FashionMNIST 的形式。

### 4.2 模型结构

配套源码中的模型层可以概括为：

```python
self.c1 = nn.Conv2d(1, 96, kernel_size=11, stride=4)
self.s2 = nn.MaxPool2d(kernel_size=3, stride=2)
self.c3 = nn.Conv2d(96, 256, kernel_size=5, padding=2)
self.s4 = nn.MaxPool2d(kernel_size=3, stride=2)
self.c5 = nn.Conv2d(256, 384, kernel_size=3, padding=1)
self.c6 = nn.Conv2d(384, 384, kernel_size=3, padding=1)
self.c7 = nn.Conv2d(384, 256, kernel_size=3, padding=1)
self.s8 = nn.MaxPool2d(kernel_size=3, stride=2)
self.f1 = nn.Linear(6*6*256, 4096)
self.f2 = nn.Linear(4096, 4096)
self.f3 = nn.Linear(4096, 10)
```

前向传播顺序：

```text
Conv1 -> ReLU -> MaxPool1
Conv2 -> ReLU -> MaxPool2
Conv3 -> ReLU -> Conv4 -> ReLU -> Conv5 -> ReLU -> MaxPool3
Flatten -> FC1 -> ReLU -> Dropout -> FC2 -> ReLU -> Dropout -> FC3
```

### 4.3 实战数据集

源码使用 `torchvision.datasets.FashionMNIST`：

- 图像本来是灰度图，所以输入通道为 1。
- 使用 `transforms.Resize(size=227)` 把图像缩放到 $227 \times 227$。
- 使用 `transforms.ToTensor()` 转成 PyTorch 张量。
- 模型输入形状是 `(B, 1, 227, 227)`。
- 输出类别为 10 类，对应 FashionMNIST 的 10 个服饰类别。

### 4.4 训练流程

训练代码的大致流程是：

1. 读取 FashionMNIST 训练集。
2. 按 8:2 划分训练集和验证集。
3. 使用 `DataLoader` 按 batch 读取数据。
4. 构建 `AlexNet()` 模型，并放到 GPU 或 CPU。
5. 使用 Adam 优化器，学习率为 0.001。
6. 使用交叉熵损失 `nn.CrossEntropyLoss()`。
7. 每个 epoch 统计训练集和验证集的 loss、accuracy。
8. 保存验证集准确率最高的模型参数。

### 4.5 测试流程

测试代码的大致流程是：

1. 构建 `AlexNet()` 模型。
2. 加载 `best_model.pth` 参数。
3. 读取 FashionMNIST 测试集。
4. 使用 `torch.no_grad()` 关闭梯度计算。
5. 前向传播得到输出 logits。
6. 使用 `torch.argmax(output, dim=1)` 得到预测类别。
7. 统计预测正确数量，计算测试准确率。

注意：训练时使用 `CrossEntropyLoss`，模型最后一层直接输出 logits 即可，不需要手动加 Softmax。

---

## 5. 注意点 Common Pitfalls

### 5.1 输入尺寸不要混淆

AlexNet 常见资料里可能出现 $224 \times 224$ 或 $227 \times 227$ 两种写法。当前课件参数详解和配套源码以 $227 \times 227$ 为准，因为：

```text
(227 - 11) / 4 + 1 = 55
```

这样第一层卷积后可以得到 $55 \times 55$。

### 5.2 原理版和实战版不要混写

- ImageNet 原理版：RGB 输入，3 通道，输出 1000 类。
- FashionMNIST 源码版：灰度图输入，1 通道，输出 10 类。

画网络结构图时，如果按课件参数页画，就写 $3 \times 227 \times 227$ 和 1000 类；如果按源码画，就写 $1 \times 227 \times 227$ 和 10 类。

### 5.3 Dropout 的代码写法

更常见的写法是定义 `nn.Dropout(p=0.5)`，或者使用：

```python
F.dropout(x, p=0.5, training=self.training)
```

这样模型在 `model.eval()` 时会自动关闭 Dropout。阅读源码时要注意 Dropout 是否跟随训练/测试模式切换。

### 5.4 AlexNet 参数很多

AlexNet 的全连接层参数量很大，尤其是 $9216 \rightarrow 4096$ 和 $4096 \rightarrow 4096$。这也是它比 LeNet 更容易过拟合、也更依赖数据增强和 Dropout 的原因之一。

---

## 6. 总结 Summary

- AlexNet 是深度卷积神经网络在大规模图像分类任务中的代表模型。
- 它继承了 LeNet 的卷积特征提取 + 全连接分类思路，但网络更深、参数更多。
- AlexNet 的主要结构是 5 个卷积层、3 个最大池化层和 3 个全连接层。
- 它使用 ReLU 替代 sigmoid / tanh，加快收敛。
- 它使用 Dropout 和数据增强来减少过拟合。
- 它使用重叠最大池化来保留更丰富的局部特征。
- 当前源码是 FashionMNIST 实战版，输入通道和输出类别数与 ImageNet 原理版不同。

***********